# Primer: worked examples

This notebook goes with [docs/primer.md](../docs/primer.md), *Jets from zero*. Each numbered cell shows one tool of task
`W1.1` of [TASKS.md](../TASKS.md). It applies the tool to a different object from the one the question uses:

- a top jet, the spray of particles from a decaying top quark;
- a QCD jet, a jet started by a single quark or gluon;
- a Z boson decaying into two quarks;
- a quark shower, the gluons that one quark radiates.

The cell shows the method and leaves the question to you. For each cell, read the primer section first, run the cell,
then answer the question named at its end (*Now try*). TASKS.md refers to these cells as cells 1 to 7 of 00_primer; cell
1 is this introduction with the imports below. The [glossary](../docs/glossary.md) lists the terms.

The notebook makes its own small samples, so it needs no `data/` folder. It runs in about ten seconds. Run it with the
kernel "Python (inside-the-jet)" ([docs/install.md](../docs/install.md)).

`import itj` alone does not import the parts of the package. Every module you use needs its own import line, as in the
cell below. Without it, Python says "module 'itj' has no attribute ...".

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import itj.observables
import itj.toy
from itj import plot
from itj.toy.kinematics import boost, delta_r, mass, pt_eta_phi, rapidity, two_body

## Cell 2: one particle as a four-vector

*Primer sections 1-3.* `itj.toy.show_ladder` makes one jet at the four levels of the toy's
[ladder](../docs/primer.md#the-toy-and-its-files), the steps by which the toy builds a jet (`partons`, `shower`,
`hadrons`, `detector`). It returns a dict of `JetData`, one jet for each level. Here we use the top jet of seed 5 and
take the hardest particle (the one with the largest transverse momentum) of its `detector` level. From its four-vector
$(p_x, p_y, p_z, E)$ (its energy and momentum together) we compute by hand its $p_T$, azimuth $\phi$, rapidity $y$ and
pseudorapidity $\eta$ ([primer](../docs/primer.md#collider-coordinates)). Then we check them against
`itj.observables.pt_y_phi_m`.

In [ ]:
top = itj.toy.show_ladder("tbqq", seed=5)  # the same top jet at the four presets: a dict of JetData
frame = top["detector"].jet(0).particles_frame()  # one row per particle, sorted by pT, hardest first
px, py, pz, e = (frame[c].to_numpy(dtype=np.float64) for c in ("part_px", "part_py", "part_pz", "part_energy"))

i = 0  # the hardest particle
pt = np.hypot(px[i], py[i])
phi = np.arctan2(py[i], px[i])  # in (-pi, pi]
y = 0.5 * np.log((e[i] + pz[i]) / (e[i] - pz[i]))
p = np.sqrt(px[i] ** 2 + py[i] ** 2 + pz[i] ** 2)
eta = 0.5 * np.log((p + pz[i]) / (p - pz[i]))  # the same as -ln tan(theta / 2)
kind = "photon" if frame["part_isPhoton"].iloc[i] else "not a photon"
print(f"particle {i} ({kind}): pT = {pt:.2f} GeV, phi = {phi:.4f}, y = {y:.4f}, eta = {eta:.4f}")

p4 = np.stack([px, py, pz, e], axis=-1)  # (particles, 4): every particle of the jet at once
pt_lib, y_lib, phi_lib, m_lib = itj.observables.pt_y_phi_m(p4)
print("the same as itj.observables.pt_y_phi_m:", np.allclose([pt, y, phi], [pt_lib[i], y_lib[i], phi_lib[i]]))

A photon is massless, so its $y$ and $\eta$ agree. Whole arrays work the same way: drop the `[i]`.

*Now try: `W1.1.2`.*

## Cell 3: the mass of a pair

*Primer section 2.* The goal is the invariant mass of a pair of particles, the mass of their summed four-vector.
`padded_p4()` turns the particles of many jets into two arrays. The array `p4` has the shape (jets, particles, 4) and is
filled with zeros after the last particle of each jet (the padding). The `mask` has the shape (jets, particles) and is
true for the real particles. Here we take one QCD jet (seed 3) at the `detector` level and compute the mass of its two
hardest particles, once exactly and once with the small-angle form $M^2 \approx E_1E_2\theta^2$.

In [ ]:
qcd = itj.toy.show_ladder("qcd", seed=3)["detector"]  # one QCD jet at the detector level
p4, mask = qcd.padded_p4()
print("p4:", p4.shape, "mask:", mask.shape, "real particles:", int(mask[0].sum()))

a, b = p4[0, 0], p4[0, 1]  # the two hardest particles of the first (and only) jet
pair = a + b
m_exact = np.sqrt(pair[3] ** 2 - pair[0] ** 2 - pair[1] ** 2 - pair[2] ** 2)
theta = np.arccos(a[:3] @ b[:3] / (np.linalg.norm(a[:3]) * np.linalg.norm(b[:3])))  # their opening angle
m_small = np.sqrt(a[3] * b[3]) * theta
print(f"opening angle {theta:.4f} rad: exact mass {m_exact:.3f} GeV, small-angle form {m_small:.3f} GeV")
print(f"the whole jet: {itj.observables.mass(p4, mask)[0]:.2f} GeV (all particles, padding masked)")

*Now try: `W1.1.1`.*

## Cell 4: a boosted decay

*Primer sections 4-6.* A Z boson (91.2 GeV) decays at rest into two massless quarks, which fly apart back to back in a
random direction (`two_body`). The whole decay is then boosted along $x$ to the velocity of a Z with $p_T$ = 500 GeV
(`boost`). The boost shows the decay as an observer sees it when the Z moves fast. We histogram the $\Delta R$ of the
two quarks, their distance in the plane of rapidity and azimuth. The mass of the pair does not change under the boost.

In [ ]:
n, m_z, pt_z = 5000, 91.2, 500.0
rng = np.random.default_rng(1)
cos, phi = rng.uniform(-1, 1, n), rng.uniform(0, 2 * np.pi, n)  # isotropic directions of quark 1 in the rest frame
q1, q2 = two_body(np.full(n, m_z), 0.0, 0.0, cos, phi)  # (n, 4) each, back to back
beta = np.array([pt_z / np.hypot(pt_z, m_z), 0.0, 0.0])  # velocity of a Z of this pT moving along x
q1, q2 = boost(q1, beta), boost(q2, beta)
print("pair mass before and after the boost:", m_z, "GeV and", np.round(mass(q1 + q2)[:3], 3), "GeV")

_, _, phi1 = pt_eta_phi(q1)
_, _, phi2 = pt_eta_phi(q2)
dr = delta_r(rapidity(q1), phi1, rapidity(q2), phi2)

fig, ax = plt.subplots(figsize=(6.0, 3.6), layout="constrained")
ax.hist(dr, bins=80, range=(0.0, 2.0), histtype="step", color=plot.palette["blue"])
ax.set_xlabel(r"$\Delta R$ between the two quarks")
ax.set_ylabel("decays per bin of 0.025")
ax.set_title(f"{n} Z decays, boosted to $p_T$ = {pt_z:.0f} GeV")
plt.show()

*Now try: `W1.1.3` (and later the stretch question `W1.1.10`).*

## Cell 5: one jet up the ladder

*Primer sections 7-8.* This cell draws the top jet of cell 2 at the four presets, the four levels of the ladder. In a
terminal, `jetgen display --ladder --process tbqq --seed 5` draws the same figure into a file. The truth record (what
the simulation knows about each particle and a real detector does not) says where each particle comes from, in two
columns. `truth_part_prong` gives the prong, the decay quark a particle comes from: 0 for the b, 1 and 2 for the two
quarks of the W. It is $-1$ for the underlying event, the soft particles from the rest of the proton-proton collision.
`truth_part_is_ue` flags the particles of the underlying event.

In [ ]:
fig = plot.ladder(top)  # the dict of cell 2
plt.show()

for level, jets in top.items():
    frame = jets.jet(0).particles_frame()
    prong = frame["truth_part_prong"].to_numpy()
    by_prong = {k: int((prong == k).sum()) for k in (0, 1, 2)}
    n_ue = int(frame["truth_part_is_ue"].sum())
    print(f"{level:9s} {len(frame):3d} particles; per prong {by_prong}; underlying event {n_ue}")

*Now try: `W1.1.4`, then `W1.1.5`.*

## Cell 6: a quark shower by hand

*Primer section 7.* `itj.toy.shower_one` is the readable version of the toy's parton shower, the repeated branching that
turns one quark or gluon into many. It showers one parton (a quark or gluon) and returns one dict per emission. Each
dict holds the angle `theta`, the momentum fraction `z`, the transverse momentum `kt` and the energy of the emitter. It
also holds the emission's `depth` in the tree and whether it is `primary` (on the hardest chain, which follows the
harder branch at each split). Here we shower a quark of 500 GeV that may radiate below the angle 0.8.

`z` is the momentum fraction of the **first** daughter (for a quark, the quark itself). It is close to 1 when the other
daughter is a soft gluon. The soft fraction of an emission is therefore $\min(z, 1 - z)$. The cell histograms it for
1 000 showers, in bins of equal width in $\ln z$.

In [ ]:
cfg = itj.toy.preset("shower").shower  # the shower settings of the presets
rng = np.random.default_rng(1)
emissions = itj.toy.shower_one(500.0, "q", 0.8, cfg, rng)
print(pd.DataFrame(emissions).round(3))

showers = [itj.toy.shower_one(500.0, "q", 0.8, cfg, rng) for _ in range(1000)]
soft = np.array([min(em["z"], 1.0 - em["z"]) for shower in showers for em in shower])
edges = np.geomspace(1e-3, 0.5, 31)
fig, ax = plt.subplots(figsize=(6.0, 3.6), layout="constrained")
ax.hist(soft, bins=edges, histtype="step", color=plot.palette["quark"])
ax.set_xscale("log")
ax.set_xlabel(r"soft fraction $\min(z, 1 - z)$")
ax.set_ylabel("emissions per bin")
ax.set_title(f"{len(soft)} emissions of 1 000 quark showers (500 GeV)")
plt.show()

*Now try: the stretch question `W1.1.8`.*

## Cell 7: a density histogram with errors

*Primer section 11.* The cell generates 3 000 jets at the `detector` level, a third each of QCD, W and top jets
(seed 7). For each process it draws the histogram of `jet_pt` as a normalised density: the entries are divided by the
number of jets and by the bin width. The error bars are the Poisson errors $\sqrt{n}$ of the counts, scaled the same
way.

In [ ]:
data = itj.toy.generate(3000, seed=7)  # a few seconds
pt = np.asarray(data["jet_pt"], dtype=np.float64)
edges = np.linspace(200.0, 1200.0, 26)  # bins of 40 GeV
centres, width = 0.5 * (edges[1:] + edges[:-1]), np.diff(edges)

fig, ax = plt.subplots(figsize=(6.5, 4.0), layout="constrained")
for process in ("qcd", "wqq", "tbqq"):
    x = pt[data.process_names == process]
    counts, _ = np.histogram(x, bins=edges)
    density = counts / (len(x) * width)  # entries per GeV: the area is 1 if no jet falls outside the bins
    error = np.sqrt(counts) / (len(x) * width)  # the Poisson error of each bin, scaled the same way
    ax.errorbar(
        centres,
        density,
        yerr=error,
        xerr=width / 2,
        fmt="o",
        ms=3,
        color=plot.palette[process],
        label=f"{process} ({len(x)} jets)",
    )
ax.set_yscale("log")
ax.set_xlabel(r"$p_T$ of the jet [GeV]")
ax.set_ylabel(r"$(1/N)\,dN/dp_T$ [1/GeV]")
ax.set_title("toy, detector level, seed 7")
ax.legend()
plt.show()

Turn this into a function that you can call for any column, process and binning. You will use it throughout the course.

*Now try: `W1.1.6`.*